# Edit Selected Mass-Spectra Plot

This notebook recreates `fig1_selected_mass_spectra.png` from the standardized mass-spectrum table. It is an editable notebook version of the `build_mass_spectra()` function in the local manuscript helper script:

```text
scripts/analysis/build_manuscript_figures.py
```

The notebook reads:

```text
data/processed/mass_spectra_standardized.csv
```

and saves edited outputs under `assets/` by default.


In [ ]:
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data" / "processed" / "mass_spectra_standardized.csv").exists():
    ROOT = Path.cwd().parents[1]

DATA = ROOT / "data" / "processed" / "mass_spectra_standardized.csv"
ASSETS = ROOT / "assets"

spectra = pd.read_csv(DATA)
print(f"Loaded {len(spectra)} rows from {DATA}")
print("Methods:", sorted(spectra["method"].unique()))
print("Energies:", sorted(spectra["energy_eV"].unique()))
spectra.head()


## Plot Controls

Edit these settings to change the plot style, selected energies, method order, labels, colors, and output filename.

In [ ]:
SELECTED_ENERGIES = [10, 40, 100]
METHODS = ["DFT-MD", "ReaxFF", "MACE-medium", "MACE-polar"]

DISPLAY = {
    "DFT-MD": "DFT/MM",
    "ReaxFF": "ReaxFF",
    "MACE-medium": "MACE-Medium",
    "MACE-polar": "MACE-Polar",
}

COLORS = {
    "DFT-MD": "#4D4D4D",
    "ReaxFF": "#009E73",
    "MACE-medium": "#0072B2",
    "MACE-polar": "#D55E00",
}

FIGSIZE = (10.5, 7.0)
DPI = 300
X_LIMIT = (0, 205)
Y_LIMIT = (0, 1.08)
LINEWIDTH = 1.8
NORMALIZE_WITHIN_PANEL = True
SHOW_PANEL_LABELS = True
SHOW_ENERGY_LABELS = True
OUTPUT_STEM = "selected_mass_spectra_edited"


In [ ]:
def configure_style():
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["Times", "Nimbus Roman", "DejaVu Serif"],
        "font.size": 14,
        "font.weight": "bold",
        "axes.linewidth": 2.0,
        "lines.linewidth": 2.0,
        "axes.labelweight": "bold",
        "axes.titleweight": "bold",
        "xtick.major.size": 6,
        "xtick.top": True,
        "ytick.right": True,
        "xtick.minor.size": 3,
        "xtick.major.width": 1.5,
        "xtick.minor.width": 1.0,
        "xtick.direction": "in",
        "ytick.major.size": 6,
        "ytick.minor.size": 3,
        "ytick.major.width": 1.5,
        "ytick.minor.width": 1.0,
        "ytick.direction": "in",
        "xtick.major.top": True,
        "xtick.minor.top": True,
        "ytick.major.right": True,
        "ytick.minor.right": True,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def add_panel_label(ax, label):
    ax.text(
        -0.12,
        1.08,
        label,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=16,
        fontweight="bold",
    )


In [ ]:
def make_mass_spectra_plot(save=True):
    configure_style()
    fig, axes = plt.subplots(
        len(SELECTED_ENERGIES),
        len(METHODS),
        figsize=FIGSIZE,
        sharex=True,
        constrained_layout=True,
    )

    if len(SELECTED_ENERGIES) == 1:
        axes = np.array([axes])

    for column, method in enumerate(METHODS):
        for row, energy in enumerate(SELECTED_ENERGIES):
            ax = axes[row, column]
            subset = spectra[
                (spectra["method"] == method)
                & (spectra["energy_eV"].astype(float) == float(energy))
            ].sort_values("mass_amu")

            values = subset["probability"].to_numpy(float)
            if NORMALIZE_WITHIN_PANEL and len(values) and values.max() > 0:
                values = values / values.max()

            ax.vlines(
                subset["mass_amu"],
                0,
                values,
                color=COLORS[method],
                linewidth=LINEWIDTH,
            )
            ax.set_ylim(*Y_LIMIT)
            ax.set_xlim(*X_LIMIT)
            ax.set_yticks([0, 1])
            ax.minorticks_on()
            ax.spines[["top", "right"]].set_visible(True)

            if SHOW_ENERGY_LABELS:
                ax.text(
                    0.96,
                    0.78,
                    f"{energy} eV",
                    transform=ax.transAxes,
                    ha="right",
                    va="center",
                    fontsize=12,
                    fontweight="bold",
                )

            if row == 0:
                ax.set_title(DISPLAY[method], fontweight="bold", pad=8)
            if column == 0 and row == len(SELECTED_ENERGIES) // 2:
                ylabel = "Relative intensity" if NORMALIZE_WITHIN_PANEL else "Occurrence"
                ax.set_ylabel(ylabel)
            if row == len(SELECTED_ENERGIES) - 1:
                ax.set_xlabel("Fragment mass (amu)")
            else:
                ax.tick_params(labelbottom=False)

    if SHOW_PANEL_LABELS:
        for column, label in enumerate("ABCDEFGHIJKLMNOPQRSTUVWXYZ"[: len(METHODS)]):
            add_panel_label(axes[0, column], label)

    if NORMALIZE_WITHIN_PANEL:
        fig.text(
            0.5,
            -0.005,
            "Peak heights are normalized within each method-energy panel.",
            ha="center",
            fontsize=11,
            fontweight="bold",
        )

    if save:
        ASSETS.mkdir(parents=True, exist_ok=True)
        png = ASSETS / f"{OUTPUT_STEM}.png"
        pdf = ASSETS / f"{OUTPUT_STEM}.pdf"
        fig.savefig(png, dpi=DPI, bbox_inches="tight")
        fig.savefig(pdf, bbox_inches="tight")
        print(f"Saved {png}")
        print(f"Saved {pdf}")
    return fig, axes

fig, axes = make_mass_spectra_plot(save=True)


## Data Inspection Helpers

Use these cells to inspect which peaks are present in each panel.

In [ ]:
# Peak counts by method and energy.
spectra.groupby(["method", "energy_eV"]).size().unstack(fill_value=0)


In [ ]:
# Edit method/energy here to inspect individual peaks.
INSPECT_METHOD = "MACE-polar"
INSPECT_ENERGY = 40

spectra[
    (spectra["method"] == INSPECT_METHOD)
    & (spectra["energy_eV"].astype(float) == float(INSPECT_ENERGY))
].sort_values("mass_amu")
